In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
import os
import shutil
from typing import Optional, List

class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir)
            if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)
        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    return "Error: Permission denied. Cannot list directory contents."

            if not os.path.exists(abs_path):
                return "Error: File not found"

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            lines = content.split("\n")

            if view_range:
                start, end = view_range
                if end == -1:
                    end = len(lines)
                selected_lines = lines[start - 1 : end]
                result = [f"{i}: {line}" for i, line in enumerate(selected_lines, start)]
            else:
                result = [f"{i}: {line}" for i, line in enumerate(lines, 1)]

            return "\n".join(result)

        except UnicodeDecodeError:
            return "Error: File contains non-text content and cannot be displayed."
        except ValueError as e:
            return f"Error: {str(e)}"
        except PermissionError:
            return "Error: Permission denied. Cannot access file."
        except Exception as e:
            return f"Error: {str(e)}"

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            return "Error: File not found"

        with open(abs_path, "r", encoding="utf-8") as f:
            content = f.read()

        match_count = self._count_matches(content, old_str)
        if match_count == 0:
            raise ValueError(
                "No match found for replacement. Please check your text and try again."
            )
        elif match_count > 1:
            raise ValueError(
                f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
            )

        self._backup_file(abs_path)
        new_content = content.replace(old_str, new_str)

        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(new_content)

        return "Successfully replaced text at exactly one location."

    def create(self, file_path: str, file_text: str) -> str:
        abs_path = self._validate_path(file_path)

        if os.path.exists(abs_path):
            raise FileExistsError("File already exists. Use str_replace to modify it.")

        os.makedirs(os.path.dirname(abs_path), exist_ok=True)

        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(file_text)

        return f"Successfully created {file_path}"

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")

        self._backup_file(abs_path)

        with open(abs_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        if lines and not lines[-1].endswith("\n"):
            new_str = "\n" + new_str

        if 0 <= insert_line <= len(lines):
            lines.insert(insert_line, new_str + "\n")
        else:
            raise IndexError(
                f"Line number {insert_line} is out of range. File has {len(lines)} lines."
            )

        with open(abs_path, "w", encoding="utf-8") as f:
            f.writelines(lines)

        return f"Successfully inserted text after line {insert_line}"

    def undo_edit(self, file_path: str) -> str:
        abs_path = self._validate_path(file_path)
        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")
        return self._restore_backup(abs_path)      

In [4]:
import json

text_editor_tool = TextEditorTool()

def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_based_edit_tool":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            # Newer tool versions may send "insert_text" instead of "new_str"
            text = tool_input.get("new_str", tool_input.get("insert_text"))
            return text_editor_tool.insert(
                tool_input["path"], tool_input["insert_line"], text
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
def get_text_edit_schema(model):
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
        "max_characters": 10000
    }

In [6]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[get_text_edit_schema(model)]
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [7]:
messages = []

add_user_message(
    messages,
    """
    Open the ./main.py file and summarize its contents in one or two sentences.
    """,
)

run_conversation(messages)


The `main.py` file defines three functions: `greeting()`, which prints "Hello, world!", and two pi calculation functions — `calculate_pi()`, which uses the Leibniz formula with 10 million iterations, and `calculate_pi_machin()`, which uses the faster-converging Machin's formula with a Taylor series for arctangent. Both pi functions return an approximation of π rounded to a specified number of decimal places.


[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    Open the ./main.py file and summarize its contents in one or two sentences.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01JLVeuSdoAUiHZn53KVeDwB',
    'name': 'str_replace_based_edit_tool',
    'input': {'command': 'view', 'path': './main.py'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01JLVeuSdoAUiHZn53KVeDwB',
    'content': '"1: def greeting():\\n2:     print(\\"Hello, world!\\")\\n3: \\n4: \\n5: def calculate_pi(precision=5):\\n6:     \\"\\"\\"\\n7:     Calculate pi to a specified number of decimal places.\\n8:     Uses the Leibniz formula: pi = 4 * (1 - 1/3 + 1/5 - 1/7 + 1/9 - ...)\\n9:     \\n10:     Args:\\n11:         precision: Number of decimal places to calculate (default: 5)\\n12:     \\n13:     Returns:\\n14:         float: Approximation of pi\\n15:     \\"\\"\\"\\n16:     pi_approx = 0.0\\n17:     iterations = 10000000  #

In [8]:
messages = []

add_user_message(
    messages,
    """
    Open the ./main.py file and write a function to calculate pi to the fifth digit.

    Then create a ./test.py file to test your implementation. 
    """,
)

run_conversation(messages)

Let me start by viewing the existing `./main.py` file first.
The file already has some existing functions. Now I'll add a clean, well-documented `calculate_pi_to_fifth_digit()` function using the **Chudnovsky algorithm** — one of the fastest converging series for π — and then create the test file.
Now let's create the `./test.py` file:
Let's do a final review of both files to make sure everything looks perfect:
Both files look great! Here's a full summary of everything that was done:

---

## `main.py` — `calculate_pi_to_fifth_digit()`

A new function was added using the **Chudnovsky algorithm**, one of the fastest-converging series ever discovered for π. Here's how it works:

| Component | Purpose |
|---|---|
| `from decimal import Decimal, getcontext` | Uses Python's arbitrary-precision `Decimal` type to avoid floating-point rounding errors during computation |
| `getcontext().prec = 20` | Sets internal precision to 20 significant digits — far more than the 5 we need |
| **Chudnovsky

[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    Open the ./main.py file and write a function to calculate pi to the fifth digit.\n\n    Then create a ./test.py file to test your implementation. \n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'Let me start by viewing the existing `./main.py` file first.'},
   {'type': 'tool_use',
    'id': 'toolu_013thsvdc8JMokivJQxNmj3b',
    'name': 'str_replace_based_edit_tool',
    'input': {'command': 'view', 'path': './main.py'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013thsvdc8JMokivJQxNmj3b',
    'content': '"1: def greeting():\\n2:     print(\\"Hello, world!\\")\\n3: \\n4: \\n5: def calculate_pi(precision=5):\\n6:     \\"\\"\\"\\n7:     Calculate pi to a specified number of decimal places.\\n8:     Uses the Leibniz formula: pi = 4 * (1 - 1/3 + 1/5 - 1/7 + 1/9 - ...)\\n9:     \\n10:     Args:\\n11:         precision: Number of decimal places to calcul